# Build a research agent with cited answers using Claude and Qdrant

Research moves faster than any model's training data. To answer questions about papers from the last few months, Claude needs to search them, and you need to check every claim it makes against a source.

This cookbook builds that agent over 10,000 recent arXiv abstracts on natural language processing, most of them from 2026. Qdrant runs semantic and keyword search in one request, and Claude cites the exact sentences that support each part of its answer.

**By the end of this cookbook, you'll be able to:**
- Index documents in Qdrant with dense vectors and a BM25 full-text index
- Run hybrid search with prefetch and Reciprocal Rank Fusion (RRF), and tune it with weights
- Return search results to Claude as `search_result` blocks, so that answers carry sentence-level citations
- Keep a multi-turn conversation that recovers from a failed turn

## Prerequisites

**Required knowledge:**
- Python fundamentals
- Basic understanding of retrieval-augmented generation (RAG)

**Required tools:**
- Python 3.11 or higher
- An Anthropic API key ([get one here](https://platform.claude.com))

**Optional:**
- A Qdrant server, version 1.20 or higher, [in Docker](https://qdrant.tech/documentation/quickstart/) or on [Qdrant Cloud](https://cloud.qdrant.io). Without one, the notebook uses Qdrant local mode, which runs in memory in this process.

## Setup

The `fastembed` extra lets the Qdrant client compute embeddings locally.

In [1]:
%%capture
%pip install -U "anthropic>=2.2.0" "qdrant-client[fastembed]>=1.20.0" datasets python-dotenv

Add your API key to a `.env` file. To use a Qdrant server, uncomment the Qdrant lines:

```
ANTHROPIC_API_KEY=your_key_here
# QDRANT_URL=http://localhost:6333
# QDRANT_API_KEY=your_qdrant_key_here
```

Without `QDRANT_URL`, the Qdrant client starts in local mode.

In [2]:
import os

import anthropic
from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

load_dotenv()

MODEL = "claude-opus-5-5"
COLLECTION = "arxiv-nlp-2026"
DENSE_MODEL = "BAAI/bge-small-en-v1.5"

claude = anthropic.Anthropic()
qdrant = QdrantClient(
    location=os.getenv("QDRANT_URL", ":memory:"),
    api_key=os.getenv("QDRANT_API_KEY"),
)

## Load recent papers

The [`librarian-bots/arxiv-metadata-snapshot`](https://huggingface.co/datasets/librarian-bots/arxiv-metadata-snapshot) dataset is a weekly copy of the arXiv metadata, released under CC0. Its rows are sorted by update date, so the first Parquet file holds the most recently updated papers. The code pins a revision, so that the data and the outputs below stay the same, and keeps 10,000 papers whose primary category is `cs.CL`. The download is about 280 MB.

In [3]:
from datasets import load_dataset

papers = load_dataset(
    "librarian-bots/arxiv-metadata-snapshot",
    revision="c428226b689a0e17db2cd55265133b9eca15d97f",
    data_files="data/train-00000-of-00011.parquet",
    columns=["id", "title", "abstract", "categories"],
    split="train",
)
papers = papers.filter(lambda row: row["categories"].split()[0] == "cs.CL").select(range(10_000))
papers[0]

{'id': '2405.00273',
 'title': 'Social Life Simulation for Non-Cognitive Skills Learning',
 'categories': 'cs.CL cs.HC',
 'abstract': 'Non-cognitive skills are crucial for personal and social life well-being, and such skill development can be supported by narrative-based (e.g., storytelling) technologies. While generative AI enables interactive and role-playing storytelling, little is known about how users engage with and perceive the use of AI in social life simulation for non-cognitive skills learning. Additionally, the benefits of AI mentorship on self-reflection awareness and ability in this context remain largely underexplored. To this end, we introduced Simulife++, an interactive platform enabled by a large language model (LLM). The system allows users to act as protagonists, creating stories with one or multiple AI-based characters in diverse social scenarios. In particular, we expanded the Human-AI interaction to a Human-AI-AI collaboration by including a Sage Agent, who acts a

## Create the collection

Each paper is searchable in two ways:

- **Semantic search** uses the named vector `dense`, which holds a `BAAI/bge-small-en-v1.5` embedding of the title and abstract.
- **Keyword search** uses a [full-text index with BM25 scoring](https://qdrant.tech/documentation/concepts/indexing/#full-text-index) on the `text` payload field. Qdrant computes the BM25 statistics itself, so there are no sparse vectors to build. This needs Qdrant 1.20 or higher.

The cell recreates the collection on every run, so a changed embedding model or an interrupted earlier run cannot leave a mismatched schema.

In [4]:
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config={
        "dense": models.VectorParams(
            size=qdrant.get_embedding_size(DENSE_MODEL),
            distance=models.Distance.COSINE,
        )
    },
)
qdrant.create_payload_index(
    collection_name=COLLECTION,
    field_name="text",
    field_schema=models.TextIndexParams(
        type=models.TextIndexType.TEXT,
        scoring=models.TextScoringParams(type=models.TextScoringType.BM25),
    ),
)
qdrant.get_collection(COLLECTION).config.params.vectors

{'dense': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None)}

On a Qdrant server, you can also add a stemmer and stopwords to the full-text index. Local mode supports only the default tokenizer.

## Index the papers

Each point holds a `models.Document` in place of a vector, and the client embeds it with FastEmbed before upload. On Qdrant Cloud, pass `cloud_inference=True` to the client and the cluster embeds instead. See [inference](https://qdrant.tech/documentation/concepts/inference/) for the supported models.

arXiv wraps titles and abstracts across lines, so the code joins the whitespace first. The payload keeps `text` for search and `abstract` for Claude to read. Point IDs must be integers or UUIDs, so each arXiv ID becomes a UUID. `wait=True` makes the count below include every point. Expect several minutes on a laptop CPU.

In [5]:
import uuid


def to_point(paper: dict) -> models.PointStruct:
    title = " ".join(paper["title"].split())
    abstract = " ".join(paper["abstract"].split())
    text = f"{title}\n\n{abstract}"
    return models.PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, paper["id"])),
        vector={"dense": models.Document(text=text, model=DENSE_MODEL)},
        payload={
            "title": title,
            "abstract": abstract,
            "text": text,
            "url": f"https://arxiv.org/abs/{paper['id']}",
        },
    )


qdrant.upload_points(
    collection_name=COLLECTION,
    points=map(to_point, papers),
    batch_size=64,
    wait=True,
)
qdrant.count(COLLECTION)

CountResult(count=10000)

## Compare dense, BM25, and hybrid search

Each method is one request in the [Query API](https://qdrant.tech/documentation/concepts/hybrid-queries/):

- **Dense** queries the `dense` vector with a `models.Document`.
- **BM25** queries the `text` index with a `models.TextQuery`.
- **Hybrid** runs both as prefetches and fuses them with `models.RrfQuery`. RRF uses only ranks, so the two score scales do not need to match. Each prefetch fetches four times the final limit to give RRF more candidates.

In [6]:
def hybrid_request(
    query: str, limit: int, weights: list[float] | None = None
) -> models.QueryRequest:
    return models.QueryRequest(
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query, model=DENSE_MODEL),
                using="dense",
                limit=limit * 4,
            ),
            models.Prefetch(query=models.TextQuery(text=query), using="text", limit=limit * 4),
        ],
        query=models.RrfQuery(rrf=models.Rrf(weights=weights)),
        limit=limit,
        with_payload=True,
    )


def hybrid_search(
    query: str, limit: int = 5, weights: list[float] | None = None
) -> list[models.ScoredPoint]:
    [response] = qdrant.query_batch_points(
        COLLECTION, requests=[hybrid_request(query, limit, weights)]
    )
    return response.points


def show(points: list[models.ScoredPoint]) -> None:
    for point in points:
        print(f"  {point.score:6.3f}  {point.payload['title']}")


def compare(query: str, limit: int = 3) -> None:
    requests = {
        "dense": models.QueryRequest(
            query=models.Document(text=query, model=DENSE_MODEL),
            using="dense",
            limit=limit,
            with_payload=True,
        ),
        "bm25": models.QueryRequest(
            query=models.TextQuery(text=query), using="text", limit=limit, with_payload=True
        ),
        "hybrid": hybrid_request(query, limit),
    }
    responses = qdrant.query_batch_points(COLLECTION, requests=list(requests.values()))
    for method, response in zip(requests, responses, strict=True):
        print(method)
        show(response.points)

Multi-head latent attention (MLA) compresses the keys and values of attention into a small latent vector. Papers usually write only the acronym:

In [7]:
compare("multi-head latent attention MLA")

dense
   0.810  Asymmetric Attention Heads: Structured Head-Wise Context Allocation for Transformer Attention
   0.802  Position Bias Correction is Insufficient for One-Pass Attention Sorting
   0.799  PRISM-$\Delta$: Differential Subspace Steering for Prompt Highlighting in Large Language Models
bm25
  25.697  An Exploratory Ablation of a Small MLA--SSM Hybrid Language Model
  21.091  Joint Subspace Sparse Attention for Efficient Long-Context Decoding
  20.297  Virtual Head Attention
hybrid
   0.600  Asymmetric Attention Heads: Structured Head-Wise Context Allocation for Transformer Attention
   0.500  An Exploratory Ablation of a Small MLA--SSM Hybrid Language Model
   0.333  Position Bias Correction is Insufficient for One-Pass Attention Sorting


- **Dense** returns three papers about attention in general. None of them mentions MLA.
- **BM25** ranks the paper with MLA in its title first, because it matches the acronym exactly.
- **Hybrid** keeps the top result of each list, so one general attention paper and the MLA paper.

For a query like this one, the exact term matters more than the meaning, so give BM25 more weight. RRF weights apply to the prefetches in order. These weights count BM25 ranks twice as much as dense ranks:

In [8]:
show(hybrid_search("multi-head latent attention MLA", limit=3, weights=[1.0, 2.0]))

   0.682  Asymmetric Attention Heads: Structured Head-Wise Context Allocation for Transformer Attention
   0.667  An Exploratory Ablation of a Small MLA--SSM Hybrid Language Model
   0.500  Joint Subspace Sparse Attention for Efficient Long-Context Decoding


Two of the three results now mention MLA, and the paper on position bias drops out.

## Return results that Claude can cite

A tool can return [`search_result` blocks](https://platform.claude.com/docs/en/build-with-claude/search-results) in place of plain text. Each block has a source, a title, and a list of text blocks. With citations on, Claude attaches each claim in its answer to the text blocks it used. Claude cites whole text blocks, so the tool splits each abstract into one block per sentence.

`@beta_tool` builds the tool schema from the type hints and the docstring. The description says that the search cannot see the conversation, which makes Claude write complete queries for short follow-up questions. The tool also prints each query that Claude sends.

In [9]:
import re

from anthropic import beta_tool


def sentences(text: str) -> list[str]:
    return re.split(r"(?<=[.!?])\s+(?=[A-Z])", text)


@beta_tool
def search_papers(query: str, limit: int = 5) -> list[dict]:
    """Search abstracts of recent arXiv papers on natural language processing.

    The search combines semantic search with BM25 keyword search. It does not see the
    conversation, so write a complete query that names the method or topic.

    Args:
        query: A complete search query, for example "KV cache compression for long context".
        limit: The number of papers to return, from 1 to 10.
    """
    print(f"search_papers: {query}")
    return [
        {
            "type": "search_result",
            "source": point.payload["url"],
            "title": point.payload["title"],
            "content": [{"type": "text", "text": s} for s in sentences(point.payload["abstract"])],
            "citations": {"enabled": True},
        }
        for point in hybrid_search(query, limit=max(1, min(limit, 10)))
    ]


search_papers.call({"query": "multi-head latent attention MLA", "limit": 1})

search_papers: multi-head latent attention MLA


[{'type': 'search_result',
  'source': 'https://arxiv.org/abs/2608.19203',
  'title': 'Asymmetric Attention Heads: Structured Head-Wise Context Allocation for Transformer Attention',
  'content': [{'type': 'text',
    'text': 'Standard multi-head attention (MHA) gives every head the same full causal context span, although heads can serve different contextual roles.'},
   {'type': 'text',
    'text': 'Some heads may rely mainly on nearby lexical or syntactic context, while others may depend on longer-range relations such as entity interactions, discourse links, or state changes.'},
   {'type': 'text',
    'text': 'We present Asymmetric Attention Heads (AAH), a head-wise context- allocation framework that treats context length as an explicit per-head or per-group allocation variable.'},
   {'type': 'text',
    'text': 'AAH groups heads using feature-derived statistics, organizes these groups hierarchically, and assigns causal local windows while preserving the standard flat MHA output in

## Run the agent with memory

A [message thread](https://platform.claude.com/docs/en/build-with-claude/message-threads) holds the conversation. `ask` adds a question and runs the [tool runner](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview), which calls `search_papers` until Claude answers without a tool call. Replies and tool results stay in the thread for the next question, and each request sends only the new messages.

A turn that fails partway leaves an unanswered question or tool call in the thread, and the API rejects the next request. So `ask` saves `thread.params` first, and on a failure, or any stop reason other than `end_turn`, it restores that state before it raises.

Other settings:

- `max_iterations=6` caps the requests for one question.
- `output_config={"effort": "medium"}` sets how much Claude thinks before it acts.
- `fallbacks="default"` reruns a [refused](https://platform.claude.com/docs/en/build-with-claude/refusals-and-fallback) request on a fallback model in the same call.

In [10]:
SYSTEM_PROMPT = (
    "You answer questions about recent NLP research. Search the paper abstracts before you "
    "answer, and answer only from the search results. If they do not answer the question, "
    "say so. Citations show the sources, so do not write paper titles or URLs. Keep answers "
    "under 200 words."
)
MAX_ITERATIONS = 6


class Conversation:
    def __init__(self, **params):
        self.thread = claude.beta.messages.thread(**params)

    def ask(self, question: str):
        checkpoint = self.thread.params
        self.thread.append({"role": "user", "content": question})
        try:
            reply = self.thread.tool_runner(max_iterations=MAX_ITERATIONS).until_done()
            if reply.stop_reason != "end_turn":
                raise RuntimeError(f"Claude stopped with stop_reason={reply.stop_reason!r}")
        except BaseException:
            self.thread = claude.beta.messages.thread(**checkpoint)
            raise
        return reply


agent = Conversation(
    model=MODEL,
    max_tokens=16000,
    system=SYSTEM_PROMPT,
    tools=[search_papers],
    output_config={"effort": "medium"},
    fallbacks="default",
    betas=["server-side-fallback-2026-07-01"],
    unsupported_request_behavior="continue_unthreaded",
)

Claude's answer arrives as text blocks, and each cited block has a list of citations. Each citation names the source and holds the cited sentence as `cited_text`. `render` numbers the sources, puts the numbers after the claims, and lists the cited sentences under each source:

In [11]:
def render(reply) -> str:
    sources: dict[tuple[str, str], list[str]] = {}
    answer = ""
    for block in reply.content:
        if block.type != "text":
            continue
        numbers = []
        for citation in block.citations or []:
            quotes = sources.setdefault((citation.title, citation.source), [])
            if citation.cited_text not in quotes:
                quotes.append(citation.cited_text)
            number = list(sources).index((citation.title, citation.source)) + 1
            if number not in numbers:
                numbers.append(number)
        answer += block.text + "".join(f" [{n}]" for n in numbers)
    lines = [answer, ""]
    for number, ((title, url), quotes) in enumerate(sources.items(), 1):
        lines.append(f"[{number}] {title} ({url})")
        lines.extend(f'    "{quote}"' for quote in quotes)
    return "\n".join(lines)

Ask the first question:

In [12]:
print(render(agent.ask("What are the newest ideas for shrinking the KV cache for long contexts?")))

search_papers: KV cache compression for long context


search_papers: KV cache quantization and token eviction for long-context LLM inference


Recent papers move past simple attention-score eviction in a few directions:

- **Smarter eviction scores:** One method keeps the cache entries whose removal would most change the model's output. It scores candidate evictions by estimating the compressed-cache logits induced by their removal [1]. Another, K-VEC, prioritizes token coverage while evicting tokens [2]. A third uses a learnable indexer that predicts KV importance [3].
- **Not throwing information away:** VarRate assigns each token a variable low-rank budget by its query salience, keeping every token at a nonzero rank [4]. ResidualKV stores a sparse set of globally retrieved references and compact, quantized residual codes [5]. SeKV keeps a lightweight summary vector on GPU [6] and stores detail on CPU so tokens can be rebuilt when needed.
- **Merging:** GRKV uses ridge-regression-based merge steps to distribute information from evicted tokens across retained tokens [7].
- **Per-head budgets:** SGD-KV favors heads that speci

Claude ran two searches, and each claim carries the number of its source. The quoted lines under each source are the exact abstract sentences that Claude cited, so you can check every claim against its evidence.

Next, a follow-up question that does not name the topic:

In [13]:
print(render(agent.ask("What about quantizing the cache instead?")))

search_papers: KV cache quantization low-bit long context


Quantization is an active area. Recent papers try to get past the limits of simple fixed-bit schemes:

- **Variable-length codes:** D-Quant notes that a $b$ bit representation is inherently limited to $2^b$ quantization levels [1], which wastes bits on the uneven spread of KV values. It uses entropy coding, plus a drift mechanism to convert entropy-coded representations of each token into fixed-size bitstreams [1] so attention kernels can still run fast.
- **Mixed and progressive precision:** For long chain-of-thought reasoning, PM-KVQ uses a progressive quantization strategy to gradually lower the bit-width of KV Cache in each block [2]. It gives more bits to more sensitive layers and reports 2.73-5.18x throughput over the original 16-bit LLMs [2].
- **Trading precision for token count:** BreadthKV spends the bytes on more tokens at low precision, combining quantization with eviction [3]. It scores above eviction alone in 17 of 18 settings [3].
- **Combining with other methods:** VFol

The search tool cannot see the conversation, yet the query names the KV cache. Claude took the topic from the thread and wrote "KV cache quantization low-bit long context".

The last question asks Claude to compare the methods it has already found:

In [14]:
print(render(agent.ask("Which of these methods work without retraining the model?")))

Only a few abstracts say outright that no training is needed. Others keep the model frozen but add small trained parts.

**Clearly training-free:**
- Behavior-Preserving Compression is a training-free framework that scores candidate evictions by estimating the compressed-cache logits induced by their removal [1].
- GRKV is a training-free KV-cache merging method [2].
- VarRate is a training-free KV codec [3]. Its authors note that comparable adaptive-rank codecs reach this allocation only through training; VarRate requires none [3].
- PM-KVQ is post-training quantization. It needs calibration data, not retraining: it leverages short calibration data with positional interpolation [4].
- BreadthKV picks its bit-width with a 60-problem end-to-end calibration [5].

**Frozen model, but small added parts are trained:**
- SeKV keeps the base LLM fully frozen and adding fewer than 0.05% trainable parameters [6].
- C²KV uses a lightweight sidecar Extractor with learnable compression tokens [7].

Claude did not search again, because the papers from the first two turns are still in the thread. It grouped the methods by what their abstracts say about training, and it named the ones whose abstracts do not say, instead of guessing.

## Next steps

In the MLA example, BM25 found the papers that use the acronym, and weighted RRF moved them up. The citations let you check each claim in an answer against the sentence it came from. To adapt this to your own data:

1. Split your documents into the units you want Claude to cite, such as sentences or paragraphs, and keep a stable source for each.
2. On a Qdrant server, add a stemmer and stopwords to the full-text index.
3. Tune the RRF `weights` and prefetch limits on real user questions.
4. Add a filter parameter to the tool, such as a date, backed by a payload index. See [filtering](https://qdrant.tech/documentation/concepts/filtering/).